## GuardGP

In [1]:
from typing import Any, Dict, List, Tuple
import pandas as pd
import torch

EPS = 1e-12
torch.set_default_dtype(torch.float64)

from models.GuardGP import run_guardgp

def run_pipeline_guardgp(
    seeds: List[int],
    outlier_types: List[str],
    outlier_ratios: List[float],
    *,
    n_total: int = 500,
    clean_prefix: int = 50,
    normal_noise: float = 0.2,
    x_range: Tuple[float, float] = (-3, 3),
    shuffle: bool = True,
    shuffle_tail_only: bool = True,
    save_csv_path: str = None,
    verbose: bool = False,
    cover_frac: float = 0.6,

    k_hist_main: int = 3,
    alpha_main: float = 0.2,
    k_hist_critic: int = 3,
    alpha_critic: float = 0.2,
   
    shard_cap_main: int = 50,
    cap_critic: int = 10,
    r_recent: int = 100,
    seed_min: int = 1,
    
    use_critic: bool = True,
    use_critic_in_fusion: bool = True,
    k_critic_seed_recent: int = 5,
    critic_seed_min: int = 10,
    critic_init_inherit_main: bool = False,
    critic_rollover_inherit: bool = True,

    
    opt_every_steps: int = 1000,
    opt_epochs_main: int = 100,
    opt_lr_main: float = 0.05,
    opt_epochs_crit: int = 100,
    opt_lr_crit: float = 0.03,
    
    use_evt_currmain: bool = True,
    use_evt_global: bool = True,


) -> Tuple[pd.DataFrame, pd.DataFrame]:
    
    rows: List[Dict[str, Any]] = []

    for otype in outlier_types:
        for ratio in outlier_ratios:
            for sd in seeds:
                if verbose:
                    print(f"=== GuardGP | type={otype} | ratio={ratio} | seed={sd} ===")

                res = run_guardgp(
                    n_total=n_total,
                    clean_prefix=clean_prefix,
                    outlier_ratio=ratio,
                    outlier_type=otype,
                    seed=sd,
                    normal_noise=normal_noise,
                    x_range=x_range,
                    shuffle=shuffle,
                    shuffle_tail_only=shuffle_tail_only,
                    verbose=False,
                    plot=False,
                    shard_cap_main=shard_cap_main,
                    cap_critic=cap_critic,
                    r_recent=r_recent,
                    seed_min=seed_min,
                    cover_frac=cover_frac,
                    k_hist_main=k_hist_main,
                    alpha_main=alpha_main,
                    k_hist_critic=k_hist_critic,
                    alpha_critic=alpha_critic,

                    use_critic=use_critic,
                    use_critic_in_fusion=use_critic_in_fusion,
                    k_critic_seed_recent=k_critic_seed_recent,
                    critic_seed_min=critic_seed_min,
                    critic_init_inherit_main=critic_init_inherit_main,
                    critic_rollover_inherit=critic_rollover_inherit,

                    opt_every_steps=opt_every_steps,
                    opt_epochs_main=opt_epochs_main,
                    opt_lr_main=opt_lr_main,
                    opt_epochs_crit=opt_epochs_crit,
                    opt_lr_crit=opt_lr_crit,
                    
                    use_evt_currmain=use_evt_currmain,
                    use_evt_global=use_evt_global,)
                rows.append(res)

    df_raw = pd.DataFrame(rows)

    group_cols = ["method", "outlier_type", "outlier_ratio"]
    df_agg = df_raw.groupby(group_cols, as_index=False).agg(
        rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"),
        smse_mean=("smse", "mean"), smse_std=("smse", "std"),
        msll_mean=("msll", "mean"), msll_std=("msll", "std"),
        nll_mean=("nll", "mean"), nll_std=("nll", "std"),
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        prec_mean=("precision", "mean"), prec_std=("precision", "std"),
        rec_mean=("recall", "mean"), rec_std=("recall", "std"),
        step_ms_mean=("avg_step_ms", "mean"), step_ms_std=("avg_step_ms", "std"),
        pred_ms_mean=("avg_pred_ms", "mean"), pred_ms_std=("avg_pred_ms", "std"),
        det_ms_mean=("avg_detect_ms", "mean"), det_ms_std=("avg_detect_ms", "std"),
        upd_ms_mean=("avg_upd_ms", "mean"), upd_ms_std=("avg_upd_ms", "std"),
    )

    if save_csv_path:
        df_raw.to_csv(save_csv_path.replace(".csv", "_raw.csv"), index=False, encoding="utf-8")
        df_agg.to_csv(save_csv_path, index=False, encoding="utf-8")

    return df_raw, df_agg


if __name__ == "__main__":
    seeds = [1,401]
    outlier_types = ["uniform", "focused", "asymmetric"]
    outlier_ratios = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8]
    shard_cap_main= 50
    cap_critic= 10
    r_recent= 100
    seed_min= 20
    
    use_critic = True
    use_critic_in_fusion = True
    k_critic_seed_recent = 10
    critic_seed_min = 10
    critic_init_inherit_main = False
    critic_rollover_inherit = True


    opt_every_steps = 1000
    opt_epochs_main = 0
    opt_lr_main = 0.05
    opt_epochs_crit = 0
    opt_lr_crit = 0.03
    
    use_evt_currmain = True
    use_evt_global = True

    df_raw, df_agg = run_pipeline_guardgp(
        seeds=seeds,
        outlier_types=outlier_types,
        outlier_ratios=outlier_ratios,
        n_total=500,
        clean_prefix=40,
        cover_frac= 0.7,
        k_hist_main= 0,
        alpha_main= 0.5,
        k_hist_critic= 0,
        alpha_critic= 0.5,
        save_csv_path="neal_GuardGP.csv",
        verbose=True,
        shard_cap_main=shard_cap_main,
        cap_critic=cap_critic,
        r_recent=r_recent,
        seed_min=seed_min,
        
        use_critic=use_critic,
        use_critic_in_fusion=use_critic_in_fusion,
        k_critic_seed_recent=k_critic_seed_recent,
        critic_seed_min=critic_seed_min,
        critic_init_inherit_main=critic_init_inherit_main,
        critic_rollover_inherit=critic_rollover_inherit,

        opt_every_steps=opt_every_steps,
        opt_epochs_main=opt_epochs_main,
        opt_lr_main=opt_lr_main,
        opt_epochs_crit=opt_epochs_crit,
        opt_lr_crit=opt_lr_crit,
        
        use_evt_currmain=use_evt_currmain,
        use_evt_global=use_evt_global,            
    )

    print("\n=== RAW (head) ===")
    print(df_raw.head())

    print("\n=== AGG (GuardGP) ===")
    print(df_agg)


=== GuardGP | type=uniform | ratio=0.1 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.1 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.2 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.2 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.3 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.3 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.4 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.4 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.5 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.5 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.6 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.6 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.7 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.7 | seed=401 ===
=== GuardGP | type=uniform | ratio=0.8 | seed=1 ===
=== GuardGP | type=uniform | ratio=0.8 | seed=401 ===
=== GuardGP | type=focused | ratio=0.1 | seed=1 ===
=== GuardGP | type=focused | ratio=0.1 | seed=401 ===
=== GuardGP | type=focused | ratio=0.2 | seed=